***Total: 42 points***

Complete this homework by writing R code to complete the following tasks. Keep in mind:

i. Empty chunks have been included where code is required
ii. This homework requires use of data files:

  - `BRCA.genome_wide_snp_6_broad_Level_3_scna.seg` (Problems 1, 2)
  - `GIAB_highconf_v.3.3.2.vcf.gz` (Problem 3)
  
iv. You will be graded on your code and output results. The assignment is worth 42 points total; partial credit can be awarded.

For additional resources, please refer to these links:  
Problems 1 & 2:  
  - https://www.bioconductor.org/packages/devel/bioc/vignettes/plyranges/inst/doc/an-introduction.html
  - https://bioconductor.org/packages/release/bioc/vignettes/GenomicRanges/inst/doc/GenomicRangesIntroduction.html  
Problem 3:  
  - https://bioconductor.org/packages/release/bioc/vignettes/Rsamtools/inst/doc/Rsamtools-Overview.pdf  
Problem 4: 
  - https://bioconductor.org/packages/release/bioc/vignettes/VariantAnnotation/inst/doc/VariantAnnotation.pdf  

# Problem 1: Overlaps between genomic regions and copy number alterations. (14 points total)

### Preparation
Load copy number segment results as shown in *2.1 BED format* of *Lecture16_GenomicData.Rmd*. You will use the same file as in the lecture notes, `BRCA.genome_wide_snp_6_broad_Level_3_scna.seg`. Here is code to get you started.

In [6]:
#load packages
suppressPackageStartupMessages({
    library(tidyverse)
    library(GenomicRanges)
    library(plyranges)
    library(VariantAnnotation)
})

In [ ]:
segs <- read.delim("BRCA.genome_wide_snp_6_broad_Level_3_scna.seg", as.is = TRUE)
mode(segs$Chromosome) <- "character" 
segs[segs$Chromosome == 23, "Chromosome"] <- "X"
segs.gr <- as(segs, "GRanges")

segs.gr %>% as_tibble()
# GRanges object (with following columns):
    # seqnames = chromosome names
    # start = start indices of sequence in chromosome (1-based indexing)
    # end = end indices of sequence in chromosome (1-based indexing)
    # width = length from start to end indices
	# strand = + or - strand (or *)
    # Sample (chr) = TCGA IDs (e.g. TCGA-3C-...)
    # Num_Probes (int) = 
    # Segment_Mean (dbl) = deletion, neutral, or gain values

seqnames,start,end,width,strand,Sample,Num_Probes,Segment_Mean
<fct>,<int>,<int>,<int>,<fct>,<chr>,<int>,<dbl>
1,3218610,95674710,92456101,*,TCGA-3C-AAAU-10A-01D-A41E-01,53225,0.0055
1,95676511,95676518,8,*,TCGA-3C-AAAU-10A-01D-A41E-01,2,-1.6636
1,95680124,167057183,71377060,*,TCGA-3C-AAAU-10A-01D-A41E-01,24886,0.0053
1,167057495,167059336,1842,*,TCGA-3C-AAAU-10A-01D-A41E-01,3,-1.0999
1,167059760,181602002,14542243,*,TCGA-3C-AAAU-10A-01D-A41E-01,9213,-0.0008
1,181603120,181609567,6448,*,TCGA-3C-AAAU-10A-01D-A41E-01,6,-1.2009
1,181610685,201473647,19862963,*,TCGA-3C-AAAU-10A-01D-A41E-01,12002,0.0055
1,201474400,201474544,145,*,TCGA-3C-AAAU-10A-01D-A41E-01,2,-1.4235
1,201475220,247813706,46338487,*,TCGA-3C-AAAU-10A-01D-A41E-01,29781,-0.0004


### a. Find the segments in `segs.gr` that have *any* overlap with the region `chr8:128,746,347-128,755,810` (4 points)
Print out the first five unique TCGA IDs.

In [ ]:
# create GRanges object for chr8:128,746,347-128,755,810 (MYC gene)
myGRange_1a <- data.frame(seqnames = "8", start = 128746347, end = 128755810) %>% as_granges()

# find overlapping segments between segs.gr and GRanges object
segs.overlap_1a <- find_overlaps(segs.gr, myGRange_1a)

segs.overlap_1a %>% 
  as_tibble() %>% 
  # select Sample (TCGA ID) column
  dplyr::select(Sample) %>%
  # choose unique TCGA IDs
  distinct(Sample) %>% 
  # slice the first five unique TCGA IDs
  dplyr::slice(1:5) %>%
  print()

# A tibble: 5 × 1
  Sample                      
  <chr>                       
1 TCGA-3C-AAAU-10A-01D-A41E-01
2 TCGA-3C-AAAU-01A-11D-A41E-01
3 TCGA-3C-AALI-10A-01D-A41E-01
4 TCGA-3C-AALI-01A-11D-A41E-01
5 TCGA-3C-AALJ-10A-01D-A41E-01


### b. Find the mean of the `Segment_Mean` values for copy number segments that have *any* overlap with the region chr17:37,842,337-37,886,915. (4 points)

In [ ]:
# create GRanges object for chr17:37,842,337-37,886,915 (ERBB2 gene)
myGRange_1b <- data.frame(seqnames = "17", start = 37842337, end = 37886915) %>% as_granges()

# find overlapping segments between segs.gr and GRanges object
segs.overlap_1b <- find_overlaps(segs.gr, myGRange_1b)

segs.overlap_1b %>% 
  as_tibble() %>% 
  # select Segment Mean column
  dplyr::select(Segment_Mean) %>%
  # calculate the mean of the values in the Segment Mean column
  summarize(meanSM = mean(Segment_Mean)) %>%
  print()

# A tibble: 1 × 1
  meanSM
   <dbl>
1  0.142


### c. Find the patient sample distribution of copy number for `PIK3CA` (hg19). (6 points)
Find the counts of samples with deletion (D; `Segment_Mean < -0.3`), neutral (N; `Segment_Mean >= -0.3 & Segment_Mean <= 0.3`), gain (G; `Segment_Mean > 0.3`) segments that have `any` overlap with `PIK3CA` gene coordinates.  


In [27]:
# look up chromosome number and start and end indices of PIK3CA gene in UCSC Genome Browser on Human (GRCh37/hg19)
# https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg19&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr3%3A178865902%2D178957881&hgsid=3426939259_PRI6CyanTdzwAkAnAafQzLrbx4io
# PIK3CA = chr3:178,865,902-178,957,881

# create GRanges object for chr3:178,865,902-178,957,881 (PIK3CA gene)
myGRange_1c <- data.frame(seqnames = "3", start = 178865902, end = 178957881) %>% as_granges()

# find overlapping segments between segs.gr and GRanges object
segs.overlap_1c <- find_overlaps(segs.gr, myGRange_1c)

segs.overlap_1c %>% 
  as_tibble() %>% 
  # create new Segment Type column classifying deletion (D), neutral (N), and gain (G) segments
  mutate(Segment_Type = case_when(Segment_Mean < -0.3 ~ "D",
                                  Segment_Mean >= -0.3 & Segment_Mean <= 0.3 ~ "N",
                                  Segment_Mean > 0.3 ~ "G")) %>%
  # group by Segment Type column
  group_by(Segment_Type) %>%
  # calculate the counts of samples with either deletion (D), neutral (N), and gain (G) in the Segment Type column
  summarize(count = n()) %>%
  print()

# A tibble: 3 × 2
  Segment_Type count
  <chr>        <int>
1 D               17
2 G              185
3 N             2026


# Problem 2: Frequency of copy number alteration events within genomic regions. (12 points total) 

This problem will continue to use the copy number data stored in `segs.gr`.

### a. Create a genome-wide tile of 1Mb windows for the human genome (`hg19`). (6 points)
See *3.1 Tiling the genome* of *Lecture16_GenomicData.Rmd* for hints.


In [ ]:
# using code from Lecture 15
seqinfo <- Seqinfo(genome = "hg19")
seqinfo <- keepStandardChromosomes(seqinfo) 
seqlevelsStyle(seqinfo) <- "NCBI"

slen <- seqlengths(seqinfo) # get the length of the chromosomes
tileWidth <- 1000000 # tile size of 1000 kb (1 Mb)
tiles <- tileGenome(seqlengths = slen, tilewidth = tileWidth,
                    cut.last.tile.in.chrom = TRUE)
tiles

Warning message in (function (seqlevels, genome, new_style) :
“cannot switch some of hg19's seqlevels from UCSC to NCBI style”


GRanges object with 3114 ranges and 0 metadata columns:
         seqnames            ranges strand
            <Rle>         <IRanges>  <Rle>
     [1]        1         1-1000000      *
     [2]        1   1000001-2000000      *
     [3]        1   2000001-3000000      *
     [4]        1   3000001-4000000      *
     [5]        1   4000001-5000000      *
     ...      ...               ...    ...
  [3110]        Y 56000001-57000000      *
  [3111]        Y 57000001-58000000      *
  [3112]        Y 58000001-59000000      *
  [3113]        Y 59000001-59373566      *
  [3114]     chrM           1-16571      *
  -------
  seqinfo: 25 sequences from an unspecified genome

### b. Find the 1Mb window with the most frequent overlapping deletions. (6 points)
Find the 1Mb windows with `any` overlap with deletion copy number segments. Assume a deletion segment is defined as a segment in `segs.gr` having `Segment_Mean < -0.3`. 

Return one of the 1Mb window `Granges` entry with the highest frequency (count) of deletion segments.

Hint: Subset the `segs.gr` to only rows with `Segment_Mean < -0.3`. 

In [53]:
# create new segs.gr object that is filtered for only deletion (D) copy number segments
segs.gr_del <- segs.gr %>% 
  as_tibble() %>%
  filter(Segment_Mean < -0.3) %>%
  GRanges()

# find overlapping segments between segs.gr_del (only deletion copy number segments) and 1Mb windows of genome-wide tile
segs.overlap_del <- find_overlaps(tiles, segs.gr_del)

# find the 1Mb window with the highest frequency of deletion segments
most_del <- segs.overlap_del %>%
  as_tibble() %>%
  group_by(seqnames, start, end) %>%
  # calculate the count of deletion segments within each 1Mb window
  summarize(count = n()) %>%
  # sort the counts in descending order so largest counts are at top
  arrange(desc(count)) %>%
  ungroup() %>%
  # slice the first (highest count) 1Mb window (note: top two windows both have 620 counts)
  dplyr::slice(1) %>%
  GRanges() %>%
  print()


`summarise()` has grouped output by 'seqnames', 'start'. You can override using
the `.groups` argument.


GRanges object with 1 range and 1 metadata column:
      seqnames            ranges strand |     count
         <Rle>         <IRanges>  <Rle> | <integer>
  [1]       16 79000001-80000000      * |       620
  -------
  seqinfo: 25 sequences from an unspecified genome; no seqlengths


# Problem 3: Reading and annotating genomic variants (16 points total)

### Preparation

In [54]:
vcfFile <- "GIAB_highconf_v.3.3.2.vcf.gz"
# https://www.bioconductor.org/packages/release/bioc/vignettes/VariantAnnotation/inst/doc/VariantAnnotation.html

### a. Load variant data from VCF file `GIAB_highconf_v.3.3.2.vcf.gz` for `chr8:128,700,000-129,000,000`. (4 points)
Note: use genome build `hg19`.

In [74]:
# using code from Lecture 16
vcfHead <- scanVcfHeader(vcfFile)

# create GRanges object for chr8:128,700,000-129,000,000 (PVT1 gene)
myGRange_3a <- data.frame(seqnames = "8", start = 128700000, end = 129000000) %>% as_granges()

vcf.param <- ScanVcfParam(which = myGRange_3a) 
vcf_3a <- readVcf(vcfFile, genome = "hg19", param = vcf.param)

### b. Combine the fields of the VCF genotype information into a table. (4 points)
You may use your choice of data objects (e.g. `data.frame`).

In [79]:
info(vcf_3a) %>%
  as.data.frame() %>%
  rownames_to_column("ID") %>%
  as_tibble()

ID,DPSum,platforms,platformnames,platformbias,datasets,datasetnames,datasetsmissingcall,callsets,callsetnames,varType,filt,callable,difficultregion,arbitrated,callsetwiththisuniqgenopassing,callsetwithotheruniqgenopassing
<chr>,<int>,<int>,<I<list>>,<I<list>>,<int>,<I<list>>,<I<list>>,<int>,<I<list>>,<chr>,<I<list>>,<I<list>>,<I<list>>,<chr>,<I<list>>,<I<list>>
rs6984323,NA,4,Illumina....,,4,HiSeqPE3....,IonExome....,5,HiSeqPE3....,NA,CS_CGnor....,CS_HiSeq....,,NA,,
rs4478537,NA,3,Illumina....,,3,HiSeqPE3....,IonExome....,4,HiSeqPE3....,NA,,CS_HiSeq....,,NA,,
rs34141920,NA,3,Illumina....,,3,HiSeqPE3....,IonExome....,4,HiSeqPE3....,NA,CS_CGnor....,CS_HiSeq....,AllRepea....,NA,,
rs17772814,NA,4,Illumina....,,5,HiSeqPE3....,IonExome,6,HiSeqPE3....,NA,CS_Solid....,CS_HiSeq....,,NA,,
rs77977256,NA,4,Illumina....,,4,HiSeqPE3....,IonExome....,5,HiSeqPE3....,NA,,CS_HiSeq....,,NA,,
8:128715845_AT/A,NA,1,Illumina,,1,HiSeqPE300x,CGnormal....,2,HiSeqPE3....,NA,CS_CGnor....,CS_HiSeq....,AllRepea....,NA,,
rs143209301,NA,3,Illumina....,,3,HiSeqPE3....,IonExome....,4,HiSeqPE3....,NA,,CS_HiSeq....,,NA,,
rs202231913,NA,1,Illumina,,1,HiSeqPE300x,CGnormal....,2,HiSeqPE3....,NA,,CS_HiSeq....,AllRepea....,NA,,
rs16902340,NA,4,Illumina....,,4,HiSeqPE3....,IonExome....,5,HiSeqPE3....,NA,,CS_HiSeq....,,NA,,


### c. Retrieve the following information at chr8:128747953. (8 points)
Print out the SNP ID (i.e. "rs ID"), reference base (`REF`), alterate base (`ALT`), genotype (`GT`), depth (`DP`), allele depth (`ADALL`), phase set (`PS`).

Hints: 

  i. `REF` and `ALT` are in the output of `rowRanges(vcf)`. See Section `3a` in `Lecture16_VariantCalls.ipynb` 
  ii. To get the sequence of `DNAString`, use `as.character(x)`.  
  ii. To get the sequence of `DNAStringSet`, use `as.character(unlist(x))`. 
  iii. To expand a list of information for `geno`, use `unlist(x)`.  

  

In [156]:
# find SNP ID corresponding to position at chr8:128,747,953
snp_id_chr8 <- rowRanges(vcf_3a) %>%
  as.data.frame() %>%
  rownames_to_column("SNP_ID") %>%
  as_tibble() %>%
  # filter for chr8:128,747,953
  filter(seqnames == 8, start == 128747953, end == 128747953) %>%
  dplyr::select(SNP_ID) %>%
  as.character()
  
GT <- geno(vcf_3a)$GT %>% 
  as.data.frame() %>%
  rownames_to_column("SNP_ID") %>%
  as_tibble() %>%
  filter(SNP_ID == snp_id_chr8) %>%
  dplyr::select(HG001) %>%
  as.character()

DP <- geno(vcf_3a)$DP %>% 
  as.data.frame() %>%
  rownames_to_column("SNP_ID") %>%
  as_tibble() %>%
  filter(SNP_ID == snp_id_chr8) %>%
  dplyr::select(HG001) %>%
  as.integer()

ADALL <- geno(vcf_3a)$ADALL %>% 
  as.data.frame() %>%
  rownames_to_column("SNP_ID") %>%
  as_tibble() %>%
  filter(SNP_ID == snp_id_chr8) %>%
  dplyr::select(HG001) %>%
  as.character() %>%
  unlist()

PS <- geno(vcf_3a)$PS %>% 
  as.data.frame() %>%
  rownames_to_column("SNP_ID") %>%
  as_tibble() %>%
  filter(SNP_ID == snp_id_chr8) %>%
  dplyr::select(HG001) %>%
  as.character()

rowRanges(vcf_3a) %>%
  as.data.frame() %>%
  rownames_to_column("SNP_ID") %>%
  as_tibble() %>%
  filter(SNP_ID == snp_id_chr8) %>%
  dplyr::select(SNP_ID, REF, ALT) %>%
  mutate(GT = GT) %>%
  mutate(DP = DP) %>%
  mutate(ADALL = ADALL) %>%
  mutate(PS = PS) %>%
  as.data.frame()

SNP_ID,REF,ALT,GT,DP,ADALL,PS
<chr>,<chr>,<I<list>>,<chr>,<int>,<chr>,<chr>
rs3824120,G,T,0|1,461,"list(rs3824120 = c(105, 94))",PATMAT
